### organizing the dataset

In [10]:
import shutil
from pathlib import Path

def reorganize_wsi(input_path: str, output_path: str):
    inp, out = Path(input_path), Path(output_path)
    (out / "masks").mkdir(parents=True, exist_ok=True)
    (out / "patches").mkdir(parents=True, exist_ok=True)

    for wsi_dir in inp.iterdir():           # iterates inp, returns Path objects
        if not wsi_dir.is_dir():
            continue
        print(wsi_dir.name)
        for mask in wsi_dir.glob("masks/*_mask.png"):
            shutil.copy2(mask, out / "masks" / mask.name.replace("_mask", ""))
        for patch in wsi_dir.glob("patches/*.png"):
            shutil.copy2(patch, out / "patches" / patch.name)

    print(f"Masks  : {len(list((out / 'masks').iterdir()))}")
    print(f"Patches: {len(list((out / 'patches').iterdir()))}")

reorganize_wsi("/data_64T_3/Raja/Alex_project/output",
               "/data_64T_3/Raja/Alex_project/dataset")

SP-24-081661 B59-1
SP-22-078912 A22-1
SP-24-081661 B54-1
SP-23-021909 B5-1
SP-22-078912 A37-1
SP-22-078912 A24-1
SP-22-078912 A27-1
SP-22-078912 A20-1
SP-24-081661 B53-1
SP-24-038839 A14-1
SP-22-078912 A33-1
SP-24-027224 B14-3
SP-22-078912 A68-1
SP-23-021909 B3-1
SP-24-081661 B55-1
SP-22-078912 A11-1
SP-22-078912 A12-1
SP-22-078912 A14-1
SP-22-078912 A15-1
SP-22-078912 A78-1
SP-24-081661 B52-1
SP-22-078912 A36-1
SP-22-078912 A25-1
S08-38628 G11
SP-24-043784 A2-1
Masks  : 1082
Patches: 1082


### Changing the groundtruth pixel values to 0 or 1

### Sampling images and masks in random

In [1]:
import os
import shutil
import random

# Define your paths
src_root = '/data_64T_3/Raja/Alex_project/dataset/training' # The folder containing masks/ and patches/
dst_root = '/data_64T_3/Raja/Alex_project/dataset/validation'

# Define subfolders
folders = ['masks', 'patches']

# 1. Get the list of common files (intersection) to ensure they match
mask_files = set(os.listdir(os.path.join(src_root, 'masks')))
patch_files = set(os.listdir(os.path.join(src_root, 'patches')))
common_files = list(mask_files.intersection(patch_files))

# 2. Sample 50 files
if len(common_files) < 50:
    print(f"Warning: Only {len(common_files)} matching files found. Sampling all.")
    samples = common_files
else:
    samples = random.sample(common_files, 50)

# 3. Create new directories and move files
for folder in folders:
    new_dir = os.path.join(dst_root, folder)
    os.makedirs(new_dir, exist_ok=True)
    
    for file_name in samples:
        src_path = os.path.join(src_root, folder, file_name)
        dst_path = os.path.join(new_dir, file_name)
        
        # Use shutil.copy to keep originals, or shutil.move to relocate them
        shutil.copy(src_path, dst_path)

print(f"Successfully moved 50 matching pairs to {dst_root}")

Successfully moved 50 matching pairs to /data_64T_3/Raja/Alex_project/dataset/validation


In [5]:
dir1 = "/data_55T_2/Raja/SEGMENTATION/masks"
dir2 = "/data_55T_2/Raja/SEGMENTATION/patches"

import os

files1, files2 = set(os.listdir(dir1)), set(os.listdir(dir2))

for label, files in [(dir1, files1 - files2), (dir2, files2 - files1)]:
    print(f"\nOnly in '{label}':" if files else "")
    [print(f"  - {f}") for f in sorted(files)]

if files1 == files2: print("Both folders have identical files.")

# Usage
compare_folders('', '')



Both folders have identical files.
Missing from : []
Missing from : []
Mismatched contents: []


In [6]:
from pathlib import Path

# Extract everything before the first '_' for each file, then convert to a set for uniqueness
unique_count = len({p.name.split('_')[0] for p in Path('/data_55T_2/Raja/SEGMENTATION/masks').glob('*.png')})

print(f"Number of unique prefixes: {unique_count}")

Number of unique prefixes: 25


In [7]:
from collections import Counter
from pathlib import Path

# Extract the prefix from every .png file and count occurrences
counts = Counter(p.name.split('_')[0] for p in Path('/data_55T_2/Raja/SEGMENTATION/masks').glob('*.png'))

print(f"Number of unique prefixes: {len(counts)}\n")
print("Images per prefix:")
for prefix, count in sorted(counts.items()):
    print(f"  {prefix}: {count}")

Number of unique prefixes: 25

Images per prefix:
  S08-38628 G11: 24
  SP-22-078912 A11-1: 125
  SP-22-078912 A12-1: 32
  SP-22-078912 A14-1: 37
  SP-22-078912 A15-1: 22
  SP-22-078912 A20-1: 9
  SP-22-078912 A22-1: 3
  SP-22-078912 A24-1: 2
  SP-22-078912 A25-1: 12
  SP-22-078912 A27-1: 5
  SP-22-078912 A33-1: 8
  SP-22-078912 A36-1: 4
  SP-22-078912 A37-1: 3
  SP-22-078912 A68-1: 22
  SP-22-078912 A78-1: 10
  SP-23-021909 B3-1: 6
  SP-23-021909 B5-1: 11
  SP-24-027224 B14-3: 66
  SP-24-038839 A14-1: 7
  SP-24-043784 A2-1: 21
  SP-24-081661 B52-1: 11
  SP-24-081661 B53-1: 71
  SP-24-081661 B54-1: 9
  SP-24-081661 B55-1: 4
  SP-24-081661 B59-1: 2


In [8]:
import csv
from collections import Counter
from pathlib import Path

# 1. Count images per prefix
folder_path = Path('/data_55T_2/Raja/SEGMENTATION/masks')
counts = Counter(p.name.split('_')[0] for p in folder_path.glob('*.png'))

# 2. Write results to a CSV file
csv_file = 'prefix_counts.csv'
with open(csv_file, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Prefix', 'Image Count'])  # Header row
    writer.writerows(sorted(counts.items()))     # Data rows sorted alphabetically

print(f"Summary saved to {csv_file} ({len(counts)} unique prefixes).")

Summary saved to prefix_counts.csv (25 unique prefixes).


### Copying patches and masks from the list 

In [1]:
import shutil
from pathlib import Path

input_files = [
    "SP-22-078912 A24-1_x027798_y027217",
    "SP-22-078912 A24-1_x027798_y027729",
    "SP-22-078912 A25-1_x012059_y075360",
    "SP-22-078912 A25-1_x012529_y074604",
    "SP-22-078912 A25-1_x012529_y075116",
    "SP-22-078912 A25-1_x012571_y074336",
    "SP-22-078912 A25-1_x012571_y074848",
    "SP-22-078912 A25-1_x012571_y075360",
    "SP-22-078912 A25-1_x013083_y074848",
    "SP-22-078912 A25-1_x013083_y075360",
    "SP-22-078912 A25-1_x013083_y075872",
    "SP-22-078912 A25-1_x013595_y074848",
    "SP-22-078912 A25-1_x013595_y075360",
    "SP-22-078912 A25-1_x014107_y074848",
    "S08-38628 G11_x003225_y008803",
    "S08-38628 G11_x003225_y009315",
    "S08-38628 G11_x003225_y009827",
    "S08-38628 G11_x003664_y013281",
    "S08-38628 G11_x003737_y009315",
    "S08-38628 G11_x003737_y009827",
    "S08-38628 G11_x003737_y010339",
    "S08-38628 G11_x003737_y010851",
    "S08-38628 G11_x004176_y013281",
    "S08-38628 G11_x004176_y013793",
    "S08-38628 G11_x004249_y009315",
    "S08-38628 G11_x004249_y009827",
    "S08-38628 G11_x004249_y010339",
    "S08-38628 G11_x004249_y010851",
    "S08-38628 G11_x004688_y013281",
    "S08-38628 G11_x004688_y013793",
    "S08-38628 G11_x004761_y009827",
    "S08-38628 G11_x004761_y010339",
    "S08-38628 G11_x004761_y010851",
    "S08-38628 G11_x005200_y013281",
    "S08-38628 G11_x005200_y013793",
    "S08-38628 G11_x005273_y010851",
    "S08-38628 G11_x005273_y011363",
    "S08-38628 G11_x005712_y013793",
    "SP-22-078912 A14-1_x051621_y015989",
    "SP-22-078912 A14-1_x051621_y016501",
    "SP-22-078912 A14-1_x052133_y015989",
    "SP-22-078912 A14-1_x052133_y016501",
    "SP-22-078912 A14-1_x052645_y016501",
    "SP-22-078912 A14-1_x053157_y016501",
    "SP-22-078912 A14-1_x053157_y017013",
    "SP-22-078912 A14-1_x053669_y017013",
    "SP-22-078912 A14-1_x054131_y017012",
    "SP-22-078912 A14-1_x054181_y017013",
    "SP-22-078912 A14-1_x054643_y017012",
    "SP-22-078912 A14-1_x054643_y017524",
    "SP-22-078912 A14-1_x055155_y017524",
    "SP-22-078912 A14-1_x055155_y018036",
    "SP-22-078912 A14-1_x069017_y063948",
    "SP-22-078912 A14-1_x069017_y064460",
    "SP-22-078912 A14-1_x069150_y063568",
    "SP-22-078912 A14-1_x069150_y064080",
    "SP-22-078912 A14-1_x070174_y064080",
    "SP-22-078912 A14-1_x070686_y063056",
    "SP-22-078912 A14-1_x070686_y063568",
    "SP-22-078912 A14-1_x070686_y064080",
    "SP-22-078912 A14-1_x071198_y063056",
    "SP-22-078912 A14-1_x071198_y063568",
    "SP-22-078912 A14-1_x071452_y063203",
    "SP-22-078912 A14-1_x071452_y063715",
    "SP-22-078912 A14-1_x071710_y063056",
    "SP-22-078912 A14-1_x071710_y063568",
    "SP-22-078912 A14-1_x071964_y062691",
    "SP-22-078912 A14-1_x071964_y063203",
    "SP-22-078912 A14-1_x071964_y063715",
    "SP-22-078912 A14-1_x072476_y063203",
    "SP-22-078912 A14-1_x072988_y062691",
    "SP-22-078912 A14-1_x073500_y062179",
    "SP-22-078912 A14-1_x073500_y062691",
    "SP-22-078912 A14-1_x074012_y061667",
    "SP-22-078912 A14-1_x074012_y062179",
    "SP-24-081661 B52-1_x084058_y045563",
    "SP-24-081661 B52-1_x084058_y046075",
    "SP-24-081661 B52-1_x084570_y045563",
    "SP-24-081661 B52-1_x084570_y046075",
    "SP-24-081661 B52-1_x085082_y045051",
    "SP-24-081661 B52-1_x085082_y045563",
    "SP-24-081661 B52-1_x085082_y046075",
    "SP-24-081661 B52-1_x085178_y045483",
    "SP-24-081661 B52-1_x085178_y045995",
    "SP-24-081661 B52-1_x085237_y045542",
    "SP-24-081661 B52-1_x085237_y046054",
]

path1, path2 = Path("/data_55T_2/Raja/SEGMENTATION/patches"), Path("/data_55T_2/Raja/SEGMENTATION/masks")
out_patches, out_masks = Path("/data_55T_2/Raja/SEGMENTATION/test/patches"), Path("/data_55T_2/Raja/SEGMENTATION/test/masks")
out_patches.mkdir(parents=True, exist_ok=True)
out_masks.mkdir(parents=True, exist_ok=True)

for name in input_files:
    for src_dir, dst_dir in [(path1, out_patches), (path2, out_masks)]:
        matches = list(src_dir.glob(f"{name}*"))
        if matches:
            for f in matches:
                shutil.copy2(f, dst_dir / f.name)
                print(f"Copied: {f} → {dst_dir / f.name}")
        else:
            print(f"Not found: {src_dir / name}")

Copied: /data_55T_2/Raja/SEGMENTATION/patches/SP-22-078912 A24-1_x027798_y027217.png → /data_55T_2/Raja/SEGMENTATION/test/patches/SP-22-078912 A24-1_x027798_y027217.png
Copied: /data_55T_2/Raja/SEGMENTATION/masks/SP-22-078912 A24-1_x027798_y027217.png → /data_55T_2/Raja/SEGMENTATION/test/masks/SP-22-078912 A24-1_x027798_y027217.png
Copied: /data_55T_2/Raja/SEGMENTATION/patches/SP-22-078912 A24-1_x027798_y027729.png → /data_55T_2/Raja/SEGMENTATION/test/patches/SP-22-078912 A24-1_x027798_y027729.png
Copied: /data_55T_2/Raja/SEGMENTATION/masks/SP-22-078912 A24-1_x027798_y027729.png → /data_55T_2/Raja/SEGMENTATION/test/masks/SP-22-078912 A24-1_x027798_y027729.png
Copied: /data_55T_2/Raja/SEGMENTATION/patches/SP-22-078912 A25-1_x012059_y075360.png → /data_55T_2/Raja/SEGMENTATION/test/patches/SP-22-078912 A25-1_x012059_y075360.png
Copied: /data_55T_2/Raja/SEGMENTATION/masks/SP-22-078912 A25-1_x012059_y075360.png → /data_55T_2/Raja/SEGMENTATION/test/masks/SP-22-078912 A25-1_x012059_y075360.pn

### Copy the images

In [2]:
import shutil
from pathlib import Path

def copy_unique_files(src_path, compare_path, dest_path):
    src, cmp, dst = Path(src_path), Path(compare_path), Path(dest_path)
    dst.mkdir(parents=True, exist_ok=True) # Ensure destination exists
    
    # Get a set of filenames in the comparison directory for fast lookup
    cmp_files = {f.name for f in cmp.iterdir() if f.is_file()}
    
    # Copy files from source to destination if they aren't in the comparison set
    for file in src.iterdir():
        if file.is_file() and file.name not in cmp_files:
            shutil.copy2(file, dst / file.name)

# Define your paths
p1 = "/data_64T_3/Raja/CDH1/annotations/2.annotations_combined/training/masks"
p2 = "/data_64T_3/Raja/CDH1/annotations/2.annotations_combined/training/patches"
p3 = "/data_64T_3/Raja/CDH1/src_segmentation/augmentation/copy_paste/masks"
p4 = "/data_64T_3/Raja/CDH1/src_segmentation/augmentation/copy_paste/patches"
p5 = "/data_64T_3/Raja/CDH1/src_segmentation/augmentation/copy_mix/masks"
p6 = "/data_64T_3/Raja/CDH1/src_segmentation/augmentation/copy_mix/patches"

# Execute comparisons and copies
# Now copying from p1 (src) filtering against p3 (cmp) into p5 (dst)
copy_unique_files(src_path=p1, compare_path=p3, dest_path=p5)
copy_unique_files(src_path=p2, compare_path=p4, dest_path=p6)

print("Copy operations completed successfully.")

Copy operations completed successfully.


In [3]:
import shutil
from pathlib import Path

def copy_unique(src_path, exclude_paths, dest_path):
    src, dst = Path(src_path), Path(dest_path)
    dst.mkdir(parents=True, exist_ok=True) # Ensure destination exists
    
    # Combine filenames from ALL exclusion paths into a single fast-lookup set
    exclude_files = {f.name for ex in exclude_paths for f in Path(ex).iterdir() if f.is_file()}
    
    # Copy files from source to destination if they aren't in the exclusion set
    for file in src.iterdir():
        if file.is_file() and file.name not in exclude_files:
            shutil.copy2(file, dst / file.name)

# Define your paths
p1 = "/data_64T_3/Raja/CDH1/annotations/2.annotations_combined/training/masks"
p2 = "/data_64T_3/Raja/CDH1/annotations/2.annotations_combined/training/patches"
p3 = "/data_64T_3/Raja/CDH1/annotations/2.annotations_combined/validation/masks"
p4 = "/data_64T_3/Raja/CDH1/annotations/2.annotations_combined/validation/patches"
p5 = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_original/training/masks"
p6 = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_original/training/patches"
p7 = "/data_64T_3/Raja/CDH1/src_segmentation/augmentation/copy_mix/masks"
p8 = "/data_64T_3/Raja/CDH1/src_segmentation/augmentation/copy_mix/patches"

# Execute comparisons and copies
# From p1, exclude if in [p3, p5], copy to p7
copy_unique(src_path=p1, exclude_paths=[p3, p5], dest_path=p7)

# From p2, exclude if in [p4, p6], copy to p8
copy_unique(src_path=p2, exclude_paths=[p4, p6], dest_path=p8)

print("Copy operations completed successfully.")

Copy operations completed successfully.


In [8]:
from pathlib import Path

def count_common_files(path1, path2):
    p1, p2 = Path(path1), Path(path2)
    
    # Create sets of filenames for both directories
    files_p1 = {f.name for f in p1.iterdir() if f.is_file()}
    files_p2 = {f.name for f in p2.iterdir() if f.is_file()}
    
    # The '&' operator finds the intersection (items present in both sets)
    common_files = files_p1 & files_p2
    
    print(f"Files in Path 1: {len(files_p1)}")
    print(f"Files in Path 2: {len(files_p2)}")
    print(f"Common files:    {len(common_files)}")
    
    return len(common_files)

# Define your paths
p1 = "/data_64T_3/Raja/CDH1/annotations/3.annotations_curated/masks"
p2 = "/data_64T_3/Raja/CDH1/annotations/4.Remaining_images_after_curated/masks" 

# Execute
count = count_common_files(p1, p2)

Files in Path 1: 526
Files in Path 2: 523
Common files:    0


In [2]:
from pathlib import Path

def remove_common_files(path1, path2):
    p1, p2 = Path(path1), Path(path2)
    
    # Create a set of filenames from p1 for instant O(1) lookup
    p1_files = {f.name for f in p1.iterdir() if f.is_file()}
    
    removed_count = 0
    
    # Iterate through p2 and delete if the name exists in p1
    for file in p2.iterdir():
        if file.is_file() and file.name in p1_files:
            file.unlink()  # WARNING: This permanently deletes the file
            removed_count += 1
            
    print(f"Successfully removed {removed_count} files from {p2.name}")

# Define your paths
p1 = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_original/test/patches"
p2 = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_original/traininng/patches"


# Execute
remove_common_files(p1, p2)

Successfully removed 86 files from patches


### Copy the images from common folder to train and test splits

In [2]:
import shutil
import pandas as pd
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
csv_file              = "/home/rajaj/Project/Alex_project/seg_framework/splits/dataset_augmented/split_train_val_test/split_train_val_test_SRC_fold1.csv"
input_patches         = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_augmentation/2.original_augmented_dataset_combined/patches/"
input_mask            = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_augmentation/2.original_augmented_dataset_combined/masks/"
output_training_patches = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_augmentation/3.original_augmented_dataset_combined_split/training/patches/"
output_training_mask    = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_augmentation/3.original_augmented_dataset_combined_split/training/masks/"
output_test_patches     = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_augmentation/3.original_augmented_dataset_combined_split/test/patches/"
output_test_mask        = "/data_64T_3/Raja/CDH1/src_segmentation/dataset_augmentation/3.original_augmented_dataset_combined_split/test/masks/"
# ─────────────────────────────────────────────────────────────────────────────

# Load test filenames (strip extensions for robust matching)
test_stems = set(Path(f).stem for f in pd.read_csv(csv_file)["test"].dropna())

def split_copy(src_dir, out_test, out_train):
    out_test, out_train = Path(out_test), Path(out_train)
    out_test.mkdir(parents=True, exist_ok=True)
    out_train.mkdir(parents=True, exist_ok=True)
    copied_test, copied_train = 0, 0
    for f in Path(src_dir).iterdir():
        if not f.is_file(): continue
        dst = out_test if f.stem in test_stems else out_train
        shutil.copy2(f, dst / f.name)
        if dst == out_test: copied_test += 1
        else: copied_train += 1
    print(f"{src_dir}: test={copied_test}  train={copied_train}")

split_copy(input_patches, output_test_patches, output_training_patches)
split_copy(input_mask,    output_test_mask,    output_training_mask)

/data_64T_3/Raja/CDH1/src_segmentation/dataset_augmentation/2.original_augmented_dataset_combined/patches/: test=86  train=618
/data_64T_3/Raja/CDH1/src_segmentation/dataset_augmentation/2.original_augmented_dataset_combined/masks/: test=86  train=618
